<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Solution: Calibration and slices

**[Calibration and slices](https://learning.nextia-ai.com/courses/math/m06/calibration-and-slices/)** · Essential Mathematics and Statistics for AI · Module 6, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** put the 1,000 test-ticket scores into bins and compare each bin's score with its share of urgent tickets, calculate recall and precision for slices by language and by team, and do the module practice: recommend a threshold with a workload limit.

| | |
|---|---|
| **Time** | About 30 minutes, after you read the lesson |
| **Needs** | Nothing to install and no account: only Python's standard library. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | The total cost (2 × flagged + 30 × missed) and the cheapest threshold, from [Thresholds and costs: The cost curve](https://learning.nextia-ai.com/courses/math/m06/thresholds-and-costs/#the-cost-curve), and the rule "review a ticket when its chance of being urgent is more than 2 ÷ 30", from [Why the best threshold moves](https://learning.nextia-ai.com/courses/math/m06/thresholds-and-costs/#the-costs). |
| **Tested** | Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m06/calibration-and-slices/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M06-L03-calibration-and-slices/calibration-and-slices-solution.ipynb).

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

## Setup: the scores of the 1,000 test tickets

The next cell defines two lists: `urgent_scores`, the model's scores of the 80 urgent test tickets, and `normal_scores`, the scores of the 920 tickets that are not urgent. They are the same seeded scores that the lesson's explorer draws. The data is fictional.

Run the cell. You should see `Ready: 80 urgent and 920 not urgent tickets, 1000 in total`.

In [ ]:
# The scores, written in full (you do not need to read them).
urgent_scores = [
    0.828, 0.694, 0.812, 0.634, 0.732, 0.326, 0.654, 0.66, 0.382, 0.674, 0.747, 0.951, 0.867, 0.507, 0.59, 0.678,
    0.469, 0.788, 0.777, 0.379, 0.701, 0.708, 0.505, 0.867, 0.58, 0.578, 0.527, 0.573, 0.684, 0.77, 0.426, 0.566,
    0.55, 0.746, 0.665, 0.538, 0.842, 0.536, 0.593, 0.655, 0.904, 0.551, 0.605, 0.766, 0.727, 0.486, 0.47, 0.693,
    0.526, 0.615, 0.895, 0.844, 0.296, 0.775, 0.308, 0.88, 0.753, 0.576, 0.565, 0.645, 0.372, 0.589, 0.446, 0.555,
    0.517, 0.579, 0.877, 0.707, 0.665, 0.356, 0.993, 0.426, 0.42, 0.633, 0.587, 0.462, 0.824, 0.813, 0.681, 0.575,
]
normal_scores = [
    0.153, 0.358, 0.566, 0.22, 0.364, 0.461, 0.137, 0.223, 0.403, 0.099, 0.213, 0.57, 0.437, 0.369, 0.431, 0.182,
    0.128, 0.525, 0.043, 0.201, 0.281, 0.36, 0.352, 0.282, 0.289, 0.033, 0.344, 0.119, 0.348, 0.586, 0.206, 0.137,
    0.048, 0.303, 0.178, 0.383, 0.478, 0.24, 0.09, 0.302, 0.332, 0.284, 0.291, 0.38, 0.19, 0.103, 0.388, 0.401,
    0.071, 0.371, 0.083, 0.23, 0.104, 0.381, 0.377, 0.048, 0.285, 0.317, 0.425, 0.38, 0.216, 0.13, 0.126, 0.501,
    0.196, 0.424, 0.195, 0.241, 0.653, 0.368, 0.418, 0.062, 0.465, 0.394, 0.455, 0.398, 0.239, 0.056, 0.085, 0.281,
    0.31, 0.255, 0.087, 0.073, 0.337, 0.116, 0.022, 0.415, 0.345, 0.412, 0.357, 0.393, 0.703, 0.524, 0.223, 0.258,
    0.303, 0.03, 0.307, 0.441, 0.153, 0.35, 0.34, 0.303, 0.127, 0.179, 0.257, 0.529, 0.142, 0.294, 0.283, 0.304,
    0.298, 0.236, 0.477, 0.108, 0.052, 0.574, 0.102, 0.456, 0.317, 0.274, 0.219, 0.464, 0.178, 0.381, 0.16, 0.168,
    0.402, 0.241, 0.302, 0.127, 0.326, 0.185, 0.291, 0.392, 0.224, 0.259, 0.443, 0.176, 0.248, 0.712, 0.269, 0.418,
    0.33, 0.084, 0.197, 0.265, 0.21, 0.02, 0.418, 0.42, 0.163, 0.488, 0.211, 0.173, 0.257, 0.258, 0.368, 0.281,
    0.048, 0.427, 0.239, 0.483, 0.158, 0.112, 0.283, 0.355, 0.414, 0.133, 0.244, 0.288, 0.281, 0.113, 0.333, 0.285,
    0.132, 0.274, 0.069, 0.134, 0.087, 0.274, 0.386, 0.468, 0.332, 0.358, 0.436, 0.255, 0.315, 0.311, 0.347, 0.119,
    0.133, 0.186, 0.435, 0.023, 0.092, 0.016, 0.643, 0.364, 0.476, 0.4, 0.056, 0.428, 0.203, 0.391, 0.255, 0.355,
    0.111, 0.102, 0.208, 0.446, 0.054, 0.287, 0.331, 0.404, 0.07, 0.47, 0.233, 0.007, 0.085, 0.32, 0.115, 0.125,
    0.443, 0.412, 0.078, 0.366, 0.186, 0.094, 0.173, 0.268, 0.139, 0.52, 0.167, 0.251, 0.306, 0.105, 0.231, 0.267,
    0.059, 0.236, 0.234, 0.315, 0.142, 0.146, 0.128, 0.228, 0.219, 0.203, 0.388, 0.147, 0.125, 0.441, 0.526, 0.063,
    0.053, 0.401, 0.34, 0.313, 0.32, 0.095, 0.242, 0.18, 0.151, 0.208, 0.312, 0.212, 0.494, 0.341, 0.346, 0.274,
    0.272, 0.321, 0.197, 0.259, 0.009, 0.165, 0.135, 0.207, 0.623, 0.26, 0.134, 0.333, 0.152, 0.316, 0.211, 0.344,
    0.129, 0.334, 0.105, 0.335, 0.176, 0.183, 0.015, 0.206, 0.162, 0.254, 0.386, 0.343, 0.595, 0.218, 0.311, 0.251,
    0.118, 0.139, 0.112, 0.253, 0.282, 0.382, 0.425, 0.426, 0.355, 0.207, 0.403, 0.307, 0.282, 0.252, 0.193, 0.213,
    0.274, 0.189, 0.398, 0.136, 0.153, 0.075, 0.416, 0.61, 0.324, 0.172, 0.212, 0.541, 0.128, 0.214, 0.368, 0.112,
    0.173, 0.108, 0.013, 0.328, 0.16, 0.194, 0.2, 0.231, 0.205, 0.142, 0.26, 0.209, 0.227, 0.118, 0.464, 0.206,
    0.259, 0.649, 0.392, 0.231, 0.202, 0.167, 0.318, 0.255, 0.383, 0.476, 0.519, 0.327, 0.289, 0.153, 0.295, 0.112,
    0.178, 0.302, 0.244, 0.345, 0.487, 0.062, 0.344, 0.296, 0.241, 0.378, 0.213, 0.433, 0.32, 0.462, 0.279, 0.177,
    0.098, 0.434, 0.175, 0.292, 0.077, 0.348, 0.387, 0.348, 0.294, 0.191, 0.169, 0.24, 0.397, 0.092, 0.379, 0.288,
    0.283, 0.308, 0.304, 0.096, 0.269, 0.075, 0.579, 0.195, 0.18, 0.269, 0.213, 0.274, 0.075, 0.069, 0.099, 0.365,
    0.209, 0.19, 0.296, 0.126, 0.066, 0.294, 0.187, 0.338, 0.095, 0.109, 0.481, 0.056, 0.236, 0.081, 0.341, 0.159,
    0.295, 0.261, 0.409, 0.33, 0.304, 0.106, 0.066, 0.267, 0.307, 0.431, 0.19, 0.299, 0.505, 0.38, 0.375, 0.366,
    0.316, 0.12, 0.471, 0.374, 0.421, 0.183, 0.346, 0.258, 0.127, 0.439, 0.308, 0.226, 0.222, 0.237, 0.214, 0.11,
    0.027, 0.197, 0.115, 0.374, 0.298, 0.433, 0.035, 0.061, 0.377, 0.289, 0.26, 0.215, 0.285, 0.006, 0.483, 0.148,
    0.097, 0.121, 0.048, 0.336, 0.152, 0.261, 0.376, 0.315, 0.186, 0.35, 0.217, 0.323, 0.193, 0.043, 0.352, 0.306,
    0.38, 0.25, 0.446, 0.579, 0.097, 0.225, 0.235, 0.462, 0.082, 0.463, 0.287, 0.149, 0.267, 0.291, 0.242, 0.297,
    0.379, 0.165, 0.375, 0.178, 0.37, 0.207, 0.162, 0.32, 0.126, 0.466, 0.139, 0.193, 0.023, 0.437, 0.422, 0.539,
    0.153, 0.297, 0.093, 0.413, 0.255, 0.282, 0.161, 0.377, 0.243, 0.237, 0.009, 0.231, 0.454, 0.274, 0.079, 0.086,
    0.089, 0.116, 0.412, 0.342, 0.324, 0.139, 0.143, 0.23, 0.173, 0.224, 0.186, 0.468, 0.304, 0.104, 0.27, 0.046,
    0.324, 0.317, 0.302, 0.306, 0.338, 0.722, 0.277, 0.248, 0.345, 0.115, 0.315, 0.368, 0.288, 0.128, 0.143, 0.337,
    0.153, 0.312, 0.144, 0.078, 0.363, 0.203, 0.226, 0.207, 0.051, 0.215, 0.184, 0.284, 0.311, 0.285, 0.1, 0.136,
    0.081, 0.221, 0.272, 0.457, 0.117, 0.373, 0.277, 0.248, 0.223, 0.264, 0.067, 0.407, 0.283, 0.34, 0.203, 0.395,
    0.179, 0.134, 0.208, 0.219, 0.352, 0.263, 0.23, 0.178, 0.31, 0.217, 0.369, 0.405, 0.311, 0.308, 0.509, 0.286,
    0.335, 0.061, 0.466, 0.282, 0.242, 0.393, 0.425, 0.08, 0.091, 0.1, 0.11, 0.35, 0.38, 0.168, 0.477, 0.204,
    0.13, 0.234, 0.298, 0.327, 0.219, 0.552, 0.329, 0.404, 0.206, 0.248, 0.353, 0.312, 0.087, 0.269, 0.242, 0.102,
    0.254, 0.337, 0.218, 0.111, 0.212, 0.297, 0.176, 0.17, 0.263, 0.413, 0.542, 0.328, 0.248, 0.304, 0.388, 0.375,
    0.332, 0.029, 0.431, 0.266, 0.025, 0.376, 0.171, 0.373, 0.216, 0.471, 0.343, 0.363, 0.522, 0.285, 0.166, 0.196,
    0.346, 0.182, 0.456, 0.362, 0.336, 0.244, 0.297, 0.151, 0.278, 0.249, 0.411, 0.158, 0.096, 0.065, 0.108, 0.074,
    0.274, 0.512, 0.373, 0.495, 0.513, 0.293, 0.311, 0.102, 0.48, 0.499, 0.339, 0.24, 0.228, 0.47, 0.271, 0.067,
    0.117, 0.3, 0.526, 0.124, 0.413, 0.229, 0.108, 0.248, 0.306, 0.168, 0.209, 0.202, 0.343, 0.36, 0.453, 0.47,
    0.239, 0.19, 0.071, 0.355, 0.125, 0.468, 0.254, 0.219, 0.304, 0.263, 0.262, 0.047, 0.52, 0.18, 0.462, 0.397,
    0.312, 0.244, 0.097, 0.333, 0.24, 0.243, 0.489, 0.28, 0.314, 0.186, 0.051, 0.356, 0.293, 0.143, 0.167, 0.172,
    0.069, 0.137, 0.303, 0.514, 0.329, 0.143, 0.109, 0.201, 0.385, 0.426, 0.204, 0.412, 0.031, 0.363, 0.062, 0.056,
    0.211, 0.348, 0.325, 0.346, 0.168, 0.37, 0.109, 0.257, 0.163, 0.455, 0.364, 0.18, 0.457, 0.162, 0.303, 0.044,
    0.22, 0.275, 0.122, 0.159, 0.352, 0.464, 0.229, 0.121, 0.4, 0.094, 0.358, 0.218, 0.509, 0.37, 0.248, 0.082,
    0.401, 0.26, 0.302, 0.236, 0.127, 0.537, 0.166, 0.009, 0.49, 0.322, 0.217, 0.144, 0.229, 0.12, 0.341, 0.029,
    0.015, 0.107, 0.707, 0.199, 0.508, 0.303, 0.372, 0.225, 0.211, 0.156, 0.217, 0.097, 0.238, 0.125, 0.385, 0.342,
    0.427, 0.146, 0.284, 0.257, 0.289, 0.109, 0.51, 0.254, 0.206, 0.038, 0.146, 0.467, 0.273, 0.519, 0.182, 0.167,
    0.247, 0.554, 0.302, 0.203, 0.411, 0.527, 0.015, 0.244, 0.289, 0.068, 0.04, 0.404, 0.244, 0.131, 0.127, 0.474,
    0.347, 0.171, 0.176, 0.322, 0.23, 0.476, 0.204, 0.221, 0.327, 0.305, 0.076, 0.728, 0.011, 0.535, 0.262, 0.135,
    0.129, 0.136, 0.212, 0.223, 0.071, 0.23, 0.606, 0.009, 0.454, 0.246, 0.198, 0.136, 0.425, 0.101, 0.312, 0.31,
    0.404, 0.24, 0.216, 0.434, 0.325, 0.115, 0.638, 0.578,
]
print(f"Ready: {len(urgent_scores)} urgent and {len(normal_scores)} not urgent tickets, {len(urgent_scores) + len(normal_scores)} in total")

## Setup: count at a threshold

The next cell defines `count(threshold)`. It returns the four counts of the confusion matrix: true positives, false negatives, false positives and true negatives. It also defines `thresholds`, the 101 thresholds from 0.00 to 1.00 in steps of 0.01.

Run the cell. You should see `TP 65, FN 15, FP 47, TN 873`: the counts at threshold 0.50 from [Classification trade-offs](https://learning.nextia-ai.com/courses/math/m06/classification-trade-offs/#worked-example-the-1000-test-tickets).

In [ ]:
def count(threshold):
    """The four counts at a threshold. A ticket is flagged when its score is at least the threshold."""
    tp = len([s for s in urgent_scores if s >= threshold])   # urgent, flagged
    fp = len([s for s in normal_scores if s >= threshold])   # not urgent, flagged
    fn = len(urgent_scores) - tp                              # urgent, missed
    tn = len(normal_scores) - fp                              # not urgent, correctly left
    return tp, fn, fp, tn

# The 101 thresholds of the explorer: 0.00, 0.01, ..., 1.00
thresholds = [round(i / 100, 2) for i in range(101)]

tp, fn, fp, tn = count(0.50)
print(f"TP {tp}, FN {fn}, FP {fp}, TN {tn}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Can you read the score as a chance?

A weather app said "70% chance of rain" on 10 days, and it rained on 7 of them. As a chance, the forecast was right. A model is **calibrated** when its scores match how often things really happen: of all tickets with a score near 0.3, about 30% should be urgent.

Priya puts the 1,000 test tickets into ten **bins** by score: 0.0 to 0.1, 0.1 to 0.2, and so on. Each bin includes its lower edge, so a score of 0.40 is in the bin 0.4 to 0.5. For each bin, she counts the tickets and the urgent tickets, and calculates the mean score.

> **Predict.** In the bin 0.4 to 0.5, the mean score is about 0.44. Is the share of urgent tickets in that bin more or less than 44%? Then run the cell.

In [ ]:
bins = [[] for _ in range(10)]          # each bin holds (score, urgent?) pairs
for s in urgent_scores:
    bins[min(9, math.floor(s * 10 + 1e-9))].append((s, True))
for s in normal_scores:
    bins[min(9, math.floor(s * 10 + 1e-9))].append((s, False))

rows = []
for i, b in enumerate(bins):
    n = len(b)
    urgent = len([1 for _, u in b if u])
    mean_score = statistics.mean(s for s, _ in b)
    rows.append([f"{i / 10:.1f} to {(i + 1) / 10:.1f}", n, urgent, f"{urgent / n:.1%}", round(mean_score, 2)])
table(["Score bin", "Tickets", "Urgent", "Share urgent", "Mean score"], rows)

> **Check.** You should see the lesson's table: 106, 198, 254, 214, 116, 57, 23, 18, 11 and 3 tickets. In the bin 0.4 to 0.5, 8 of 116 tickets are urgent (6.9%), with a mean score of 0.44.

The lesson's **calibration diagram** draws one dot per bin. The next cell shows the same comparison as text: for each bin, the mean score and the observed share, both as bars of up to 50 characters. If the observed bar is much shorter than the score bar, the model is **overconfident** in that bin. If it is longer, the model is **underconfident**.

> **Predict.** In which bins is the model overconfident? Then run the cell.

In [ ]:
for i, b in enumerate(bins):
    mean_score = statistics.mean(s for s, _ in b)
    observed = len([1 for _, u in b if u]) / len(b)
    gap = observed - mean_score
    verdict = "overconfident" if gap < -0.1 else "underconfident" if gap > 0.1 else "close"
    print(f"{i / 10:.1f}–{(i + 1) / 10:.1f}  score    {'█' * round(mean_score * 50)} {mean_score:.2f}")
    print(f"{'':7}  observed {'▒' * round(observed * 50)} {observed:.2f}   {verdict} ({len(b)} tickets)")

> **Check.** You should see "overconfident" for the five bins from 0.1 to 0.6, "close" for 0.6 to 0.8, and "underconfident" for 0.8 to 0.9. The first bin is "close" only because its scores are small (0.06). The last two bins have only 11 and 3 tickets, so do not draw a conclusion from them. The cell calls a gap of more than 0.1 not "close": that limit is a choice for this notebook, not a rule.

The model **ranks** tickets well: the observed share almost always rises with the score. The cost curve used only the counts above and below each threshold, so the threshold from the last lesson is still valid. But a score of 0.44 does not mean "44% likely urgent".

## 2. The 2 ÷ 30 rule needs real chances

In the last lesson, a review paid for itself when the chance that a ticket is urgent is more than 2 ÷ 30. Run the cell. You should see a limit of 6.7%, and the bin 0.4 to 0.5 at 6.9%: almost exactly at the limit. That is why the cheapest threshold, 0.42, is inside this bin.

In [ ]:
b = bins[4]
print(f"limit: {2 / 30:.1%}")
print(f"bin 0.4 to 0.5: {len([1 for _, u in b if u]) / len(b):.1%} urgent")

## 3. One average hides a weak group

At threshold **0.44**, the model catches 70 of the 80 urgent tickets. Run the cell. You should see TP 70 and a recall of 87.5%.

In [ ]:
tp, fn, fp, tn = count(0.44)
print(f"TP {tp}, FN {fn}, recall {tp / (tp + fn):.1%}")

Mei splits the test set by language. A part of the data chosen by one property is a **slice**. The slice counts are in the lesson's table: the scores in the setup do not record the language.

> **Predict.** The overall recall is 87.5%. Is the recall for tickets in other languages close to it? Then run the cell.

In [ ]:
language = {  # slice: (tickets, urgent, urgent and flagged), at threshold 0.44
    "English": (850, 64, 62),
    "Other languages": (150, 16, 8),
}
rows = [[name, n, u, f, f"{f / u:.1%}"] for name, (n, u, f) in language.items()]
rows.append(["All tickets", 1000, 80, 70, f"{70 / 80:.1%}"])
table(["Slice", "Tickets", "Urgent", "Urgent and flagged", "Recall"], rows)
print()
bars(["All tickets", "English", "Other languages"], [87.5, round(62 / 64 * 100, 1), 50.0])

> **Check.** You should see a recall of 96.9% for English and 50.0% for other languages. The overall 87.5% is close to English, because English tickets are 85% of the test set. A customer who writes in another language has only a 50% chance that their urgent ticket reaches a senior agent.

## 4. Guided practice: slices by team

Grace wants the same check for her five teams, at threshold 0.44.

> **Your turn.** Replace each `...`, then run the check cell. Write one sentence for Grace in `c03-notes.md`: what you see, and how sure you are.
>
> - `total_urgent` and `total_flagged`: the sums of the two columns.
> - `recall_by_team`: a dictionary from team name to recall in percent, for example `{"Payment": 96.7, ...}`.
> - `lowest_team`: the name of the team with the lowest recall.
> - `one_ticket`: how many percentage points one more caught ticket adds to that team's recall.

In [ ]:
teams = {  # team: (urgent tickets, urgent and flagged)
    "Payment": (30, 29),
    "Delivery": (22, 20),
    "Returns": (12, 10),
    "Warranty": (8, 6),
    "Account": (8, 5),
}

total_urgent = sum(u for u, f in teams.values())
total_flagged = sum(f for u, f in teams.values())
recall_by_team = {team: f / u * 100 for team, (u, f) in teams.items()}
lowest_team = min(recall_by_team, key=recall_by_team.get)
one_ticket = 1 / teams[lowest_team][0] * 100

Run the check cell.

In [ ]:
expect("the total of urgent tickets", total_urgent, 80)
expect("the total of urgent and flagged tickets", total_flagged, 70)
if recall_by_team is ...:
    expect("the recall of each team", recall_by_team, None)
else:
    for team, expected in [("Payment", 96.7), ("Delivery", 90.9), ("Returns", 83.3), ("Warranty", 75.0), ("Account", 62.5)]:
        expect(f"the recall of {team}", recall_by_team.get(team, ...), expected, tol=0.05)
expect("the team with the lowest recall", lowest_team, "Account")
expect("the change from one ticket, in points", one_ticket, 12.5)

The Account team misses 3 of 8 urgent tickets, but 8 tickets are too few to be sure: one ticket moves its recall by 12.5 points. The language gap (50% against 97%) is far too big to come from a few tickets. The account gap could come from one or two.

## 5. Independent variation: precision by language

At threshold 0.44, the model flags 170 tickets. 145 are English, and 62 of them are urgent. 25 are in other languages, and 8 of them are urgent.

> **Your turn.** Calculate the precision for each slice and for all tickets, in percent. Then run the check cell.

In [ ]:
precision_english = 62 / 145 * 100
precision_other = 8 / 25 * 100
precision_all = (62 + 8) / (145 + 25) * 100

Run the check cell.

In [ ]:
expect("the precision for English", precision_english, 42.8, tol=0.05)
expect("the precision for other languages", precision_other, 32.0, tol=0.05)
expect("the precision for all tickets", precision_all, 41.2, tol=0.05)

For tickets in other languages, the model has a lower recall **and** a lower precision. A senior agent who opens a flagged ticket in another language finds a false alarm about 2 times in 3. That points to the training data, not to the threshold.

## 6. Module practice

### Worked example: a threshold with a workload limit

Omar's costs are 2 points for a review and 30 for a missed urgent ticket. Grace's senior agents can review **at most 30 tickets a day**.

**Step 1.** Turn the limit into flags per 1,000 test tickets: 30 a day × 7 = 210 a week, and 210 ÷ 1.2 = 175. Run the cell. You should see `175.0`.

In [ ]:
limit = 30 * 7 / 1.2
print(limit)

**Step 2.** List the thresholds near the bottom of the cost curve, with the reviews per day and the cost. Run the cell.

> **Predict.** Is the cheapest threshold, 0.42, inside the limit?

In [ ]:
def practice_table(miss):
    rows = []
    for t in [0.40, 0.42, 0.44, 0.45, 0.46, 0.48, 0.50]:
        tp, fn, fp, tn = count(t)
        flagged = tp + fp
        rows.append([f"{t:.2f}", flagged, round(flagged * 1.2 / 7, 1), fn, 2 * flagged + miss * fn,
                     "yes" if flagged <= limit else "no"])
    table(["Threshold", "Flagged", "Reviews per day", "Missed", "Total cost", "Inside the limit?"], rows, digits=1)

practice_table(miss=30)

> **Check.** You should see 0.40 (666) and 0.42 (606) outside the limit, with 39.1 and 33.9 reviews a day. Inside the limit, 0.44 costs 640 and 0.46 costs 638, with 29.1 and 26.4 reviews a day.

**Step 3.** 0.44 and 0.46 cost almost the same: 2 points per 1,000 tickets is much less than the uncertainty from 80 urgent tickets. Priya chooses 0.44, because it misses one urgent ticket fewer (10 against 11). Her full recommendation is on the lesson page.

### Your task: the costs change

Omar changes the cost of a missed urgent ticket to **60 cost points**. A review still costs 2, and the limit is still 30 reviews a day (175 flags per 1,000 tickets).

> **Your turn.** Run `practice_table(miss=60)` in the first cell if you want to see the table. Then replace each `...` and run the check cell. Write your recommendation of three to five sentences for Omar in `c03-notes.md`.
>
> - `best_inside`: the cheapest threshold from 0.00 to 1.00 with at most 175 flags, and `cost_inside`: its total cost.
> - `best_any`: the cheapest threshold without the limit, and `cost_any`: its total cost.

In [ ]:
practice_table(miss=60)

def cost_60(t):
    tp, fn, fp, tn = count(t)
    return 2 * (tp + fp) + 60 * fn

def flagged(t):
    tp, fn, fp, tn = count(t)
    return tp + fp

inside = [t for t in thresholds if flagged(t) <= limit]
best_inside = min(inside, key=lambda t: (cost_60(t), t))
cost_inside = cost_60(best_inside)
best_any = min(thresholds, key=lambda t: (cost_60(t), t))
cost_any = cost_60(best_any)

Run the check cell.

In [ ]:
expect("the cheapest threshold inside the limit", best_inside, 0.44)
expect("its total cost", cost_inside, 940)
expect("the cheapest threshold without the limit", best_any, 0.37)
expect("its total cost", cost_any, 802)

Inside the limit, 0.44 is still the cheapest choice, at 940 points per 1,000 tickets. Without the limit, 0.37 costs 802 points but needs about 48 reviews a day. A good recommendation uses both costs, respects the limit, shows the calculation, and says what changes if the limit changes. The model answer is on the lesson page.

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/math/m06/calibration-and-slices/#module-check). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook you put the scores into ten bins and saw that the model is overconfident from 0.2 to 0.5: a score near 0.44 means about 7% urgent, not 44%. The model still ranks the tickets well, so the cheapest threshold stays valid. At threshold 0.44, recall is 87.5% overall but 50% for tickets in other languages. With a limit of 30 reviews a day, 0.44 is the cheapest threshold inside the limit, for a miss at 30 and at 60 points.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/math/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m06/calibration-and-slices/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people.